---
## Setup: Create Database and Load Sample Data

In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect("ecommerce.db")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
 customer_id INTEGER PRIMARY KEY,
 first_name TEXT NOT NULL,
 last_name TEXT NOT NULL,
 email TEXT UNIQUE NOT NULL,
 city TEXT NOT NULL,
 state TEXT NOT NULL,
 join_date DATE NOT NULL,
 is_premium BOOLEAN DEFAULT FALSE
);

CREATE TABLE products (
 product_id INTEGER PRIMARY KEY,
 product_name TEXT NOT NULL,
 category TEXT NOT NULL,
 brand TEXT NOT NULL,
 unit_price REAL NOT NULL CHECK(unit_price > 0),
 stock_qty INTEGER NOT NULL DEFAULT 0 CHECK(stock_qty >= 0)
);

CREATE TABLE orders (
 order_id INTEGER PRIMARY KEY,
 customer_id INTEGER NOT NULL,
 order_date DATE NOT NULL,
 status TEXT NOT NULL DEFAULT 'Pending',
 total_amount REAL NOT NULL CHECK(total_amount >= 0),
 FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
 item_id INTEGER PRIMARY KEY,
 order_id INTEGER NOT NULL,
 product_id INTEGER NOT NULL,
 quantity INTEGER NOT NULL CHECK(quantity > 0),
 unit_price REAL NOT NULL CHECK(unit_price > 0),
 discount_pct REAL DEFAULT 0 CHECK(discount_pct BETWEEN 0 AND 100),
 FOREIGN KEY(order_id) REFERENCES orders(order_id),
 FOREIGN KEY(product_id) REFERENCES products(product_id)
);

INSERT INTO customers VALUES
(101,'Aarav','Sharma','aarav.s@email.com','Mumbai','Maharashtra','2024-01-15',1),
(102,'Priya','Patel','priya.p@email.com','Ahmedabad','Gujarat','2024-02-20',0),
(103,'Rohan','Gupta','rohan.g@email.com','Delhi','Delhi','2024-03-10',1),
(104,'Sneha','Reddy','sneha.r@email.com','Hyderabad','Telangana','2024-04-05',0),
(105,'Vikram','Singh','vikram.s@email.com','Jaipur','Rajasthan','2024-05-12',1),
(106,'Ananya','Iyer','ananya.i@email.com','Chennai','Tamil Nadu','2024-06-18',0),
(107,'Karan','Mehta','karan.m@email.com','Pune','Maharashtra','2024-07-22',1),
(108,'Divya','Nair','divya.n@email.com','Kochi','Kerala','2024-08-30',0);

INSERT INTO products VALUES
(201,'Wireless Earbuds','Electronics','BoAt',1499,250),
(202,'Cotton T-Shirt','Clothing','Levis',799,500),
(203,'Smart Watch','Electronics','Noise',2999,150),
(204,'Running Shoes','Clothing','Nike',4599,120),
(205,'Bluetooth Speaker','Electronics','JBL',3499,200),
(206,'Bedsheet Set','Home','Spaces',1299,300),
(207,'Laptop Stand','Electronics','AmazonBasics',899,180),
(208,'Cushion Covers (Set)','Home','HomeCenter',599,400);

INSERT INTO orders VALUES
(1001,101,'2024-08-01','Delivered',4498),
(1002,102,'2024-08-03','Delivered',799),
(1003,103,'2024-08-05','Shipped',7498),
(1004,101,'2024-08-10','Delivered',3499),
(1005,104,'2024-08-12','Cancelled',2999),
(1006,105,'2024-08-15','Delivered',5898),
(1007,106,'2024-08-18','Pending',1299),
(1008,103,'2024-08-20','Delivered',899),
(1009,107,'2024-08-25','Shipped',6098),
(1010,108,'2024-08-28','Delivered',1598);

INSERT INTO order_items VALUES
(5001,1001,201,2,1499,0),(5002,1001,207,1,899,10),(5003,1002,202,1,799,0),
(5004,1003,203,1,2999,0),(5005,1003,204,1,4599,5),(5006,1004,205,1,3499,0),
(5007,1005,203,1,2999,0),(5008,1006,201,1,1499,10),(5009,1006,204,1,4599,5),
(5010,1007,206,1,1299,0),(5011,1008,207,1,899,0),(5012,1009,205,1,3499,0),
(5013,1009,208,2,599,15),(5014,1010,206,1,1299,0),(5015,1010,208,1,599,0);
""")
conn.commit()
print("Database ready")

Database ready


---
# Section A – SQL Basics


## Q1 – SELECT all customers

In [ ]:
pd.read_sql_query("""SELECT * FROM customers;""", conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


## Q2 – SELECT first_name, last_name, city from customers

In [ ]:
pd.read_sql_query("""SELECT first_name, last_name, city FROM customers;""", conn)

,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


## Q3 – DISTINCT product categories

In [ ]:
pd.read_sql_query("""SELECT DISTINCT category FROM products;""", conn)

,category
0,Electronics
1,Clothing
2,Home


## Q4 – Primary Keys

| Table | Primary Key |
|-------------|-------------|
| customers | customer_id |
| products | product_id |
| orders | order_id |
| order_items | item_id |

A **Primary Key** must satisfy two rules:
- **Unique** – No two rows can share the same value.
- **NOT NULL** – Every row must have an identifier; the column cannot be empty.

This guarantees that every record in the table can be uniquely identified and referenced.

## Q5 – Constraints on the `email` column

```sql
email VARCHAR(100) UNIQUE NOT NULL
```

Two constraints are applied:
1. **UNIQUE** – No two customers can register with the same email address.
2. **NOT NULL** – Every customer must provide an email; the field cannot be left empty.

If a duplicate email is inserted, the database raises a **UNIQUE constraint violation** error and rejects the INSERT.

## Q6 – Invalid Product Insert Example

```sql
INSERT INTO products (product_id, product_name, category, brand, unit_price, stock_qty)
VALUES (209, 'Test Product', 'Electronics', 'TestBrand', -50, 100);
```

This statement **fails** because the `products` table has the constraint:

```sql
unit_price REAL NOT NULL CHECK(unit_price > 0)
```

A value of `-50` violates `CHECK(unit_price > 0)`, so the database raises a **CHECK constraint violation** error and the row is not inserted.

---
# Section B – Filtering & WHERE Clause


## Q7 – Retrieve all orders with status = 'Delivered'. 

In [ ]:
query = """
SELECT *
FROM orders
WHERE status = 'Delivered';
"""

pd.read_sql_query(query, conn)

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498.0
1,1002,102,2024-08-03,Delivered,799.0
2,1004,101,2024-08-10,Delivered,3499.0
3,1006,105,2024-08-15,Delivered,5898.0
4,1008,103,2024-08-20,Delivered,899.0
5,1010,108,2024-08-28,Delivered,1598.0


## Q8 – Electronics products with price > ₹2000

In [ ]:
pd.read_sql_query("""SELECT * FROM products WHERE category='Electronics' AND unit_price>2000;""", conn)

,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999.0,150
1,205,Bluetooth Speaker,Electronics,JBL,3499.0,200


## Q9 – Maharashtra customers who joined in 2024

In [ ]:
pd.read_sql_query("""SELECT * FROM customers WHERE state='Maharashtra' AND join_date BETWEEN '2024-01-01' AND '2024-12-31';""", conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1


## Q10 – Orders between Aug 10–25, 2024 (excluding Cancelled)

In [ ]:
pd.read_sql_query("""SELECT * FROM orders WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25' AND status<>'Cancelled';""", conn)

,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499.0
1,1006,105,2024-08-15,Delivered,5898.0
2,1007,106,2024-08-18,Pending,1299.0
3,1008,103,2024-08-20,Delivered,899.0
4,1009,107,2024-08-25,Shipped,6098.0


## Q11 – Purpose of `idx_orders_date`

An index on `order_date` speeds up **filtering and sorting** queries that involve that column.

**Example query that benefits:**
```sql
SELECT * FROM orders
WHERE order_date BETWEEN '2024-08-01' AND '2024-08-31';
```

Without the index, the database performs a **full table scan** (checks every row).  
With `idx_orders_date`, it uses the **B-tree index** to jump directly to matching rows, dramatically reducing I/O for large tables.

## Q12 – SARGable vs Non-SARGable Query

**Non-SARGable (index NOT used – wraps column in a function):**
```sql
SELECT * FROM customers
WHERE YEAR(join_date) = 2024;
```
The function `YEAR()` is applied to every row, so the index on `join_date` cannot be used.

**SARGable / Index-friendly rewrite:**
```sql
SELECT * FROM customers
WHERE join_date BETWEEN '2024-01-01' AND '2024-12-31';
```
The column is compared directly against constants, allowing the database engine to use a range scan on the index.

**Rule:** Never wrap an indexed column in a function inside a `WHERE` clause if you want the index to be used.

---
# Section C – Aggregates & GROUP BY


## Q13 – Total number of orders

In [ ]:
pd.read_sql_query("""SELECT COUNT(*) AS total_orders FROM orders;""", conn)

,total_orders
0,10


## Q14 – Total revenue from Delivered orders

In [ ]:
pd.read_sql_query("""SELECT SUM(total_amount) AS revenue FROM orders WHERE status='Delivered';""", conn)

,revenue
0,17191.0


## Q15 – Average price per product category

In [ ]:
pd.read_sql_query("""SELECT category, AVG(unit_price) avg_price FROM products GROUP BY category;""", conn)

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


## Q16 – Order count and revenue by status

In [ ]:
pd.read_sql_query("""SELECT status, COUNT(*) order_count, SUM(total_amount) revenue FROM orders GROUP BY status ORDER BY revenue DESC;""", conn)

,status,order_count,revenue
0,Delivered,6,17191.0
1,Shipped,2,13596.0
2,Cancelled,1,2999.0
3,Pending,1,1299.0


## Q17 – MAX and MIN price per category

In [ ]:
pd.read_sql_query("""SELECT category, MAX(unit_price) most_expensive, MIN(unit_price) cheapest FROM products GROUP BY category;""", conn)

,category,most_expensive,cheapest
0,Clothing,4599.0,799.0
1,Electronics,3499.0,899.0
2,Home,1299.0,599.0


## Q18 – Categories with average price > ₹2000 (HAVING)

In [ ]:
pd.read_sql_query("""SELECT category, AVG(unit_price) avg_price FROM products GROUP BY category HAVING AVG(unit_price)>2000;""", conn)

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0


---
# Section D – JOINs & Relationships


## Q19 – Orders with customer names (INNER JOIN)

In [ ]:
pd.read_sql_query("""SELECT o.order_id, o.order_date, c.first_name, c.last_name, o.total_amount FROM orders o JOIN customers c ON o.customer_id=c.customer_id;""", conn)

,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498.0
1,1002,2024-08-03,Priya,Patel,799.0
2,1003,2024-08-05,Rohan,Gupta,7498.0
3,1004,2024-08-10,Aarav,Sharma,3499.0
4,1005,2024-08-12,Sneha,Reddy,2999.0
5,1006,2024-08-15,Vikram,Singh,5898.0
6,1007,2024-08-18,Ananya,Iyer,1299.0
7,1008,2024-08-20,Rohan,Gupta,899.0
8,1009,2024-08-25,Karan,Mehta,6098.0
9,1010,2024-08-28,Divya,Nair,1598.0


## Q20 – All customers with their orders (LEFT JOIN)

In [ ]:
pd.read_sql_query("""SELECT c.*, o.order_id, o.order_date FROM customers c LEFT JOIN orders o ON c.customer_id=o.customer_id;""", conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium,order_id,order_date
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1,1001,2024-08-01
1,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1,1004,2024-08-10
2,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0,1002,2024-08-03
3,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1,1003,2024-08-05
4,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1,1008,2024-08-20
5,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0,1005,2024-08-12
6,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1,1006,2024-08-15
7,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0,1007,2024-08-18
8,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1,1009,2024-08-25
9,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0,1010,2024-08-28


## Q21 – Order items with product names (3-table JOIN)

In [ ]:
pd.read_sql_query("""SELECT o.order_id, p.product_name, oi.quantity, oi.unit_price, oi.discount_pct FROM orders o JOIN order_items oi ON o.order_id=oi.order_id JOIN products p ON oi.product_id=p.product_id;""", conn)

,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499.0,0.0
1,1001,Laptop Stand,1,899.0,10.0
2,1002,Cotton T-Shirt,1,799.0,0.0
3,1003,Smart Watch,1,2999.0,0.0
4,1003,Running Shoes,1,4599.0,5.0
5,1004,Bluetooth Speaker,1,3499.0,0.0
6,1005,Smart Watch,1,2999.0,0.0
7,1006,Wireless Earbuds,1,1499.0,10.0
8,1006,Running Shoes,1,4599.0,5.0
9,1007,Bedsheet Set,1,1299.0,0.0


## Q22 – LEFT JOIN vs RIGHT JOIN vs FULL OUTER JOIN

| Join Type | What it returns |
|-----------------|--------------------------------------------------|
| **LEFT JOIN** | All rows from the **left** table + matching rows from the right. Non-matching right rows appear as NULL. |
| **RIGHT JOIN** | All rows from the **right** table + matching rows from the left. Non-matching left rows appear as NULL. |
| **FULL OUTER JOIN** | All rows from **both** tables. NULLs fill in wherever there is no match on either side. |

**Example:** `customers LEFT JOIN orders` returns every customer, even those who have never placed an order (their `order_id` columns will be NULL).

## Q23 – Foreign Key Relationships

```
orders.customer_id      → customers.customer_id
order_items.order_id    → orders.order_id
order_items.product_id  → products.product_id
```

**What happens if you try to insert an order with a non-existent customer?**

```sql
INSERT INTO orders VALUES (2000, 999, '2024-09-01', 'Pending', 1000);
```

The database **rejects** this INSERT with a **FOREIGN KEY constraint violation** because `customer_id = 999` does not exist in the `customers` table. This referential integrity check ensures that no orphan records can be created.

---
# Section E – CASE Expressions


## Q24 – Product price tier using CASE

In [ ]:
pd.read_sql_query("""
SELECT product_name, unit_price,
  CASE
    WHEN unit_price < 1000 THEN 'Budget'
    WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
    ELSE 'Premium'
  END AS price_tier
FROM products;
""", conn)

,product_name,unit_price,price_tier
0,Wireless Earbuds,1499.0,Mid-Range
1,Cotton T-Shirt,799.0,Budget
2,Smart Watch,2999.0,Mid-Range
3,Running Shoes,4599.0,Premium
4,Bluetooth Speaker,3499.0,Premium
5,Bedsheet Set,1299.0,Mid-Range
6,Laptop Stand,899.0,Budget
7,Cushion Covers (Set),599.0,Budget


## Q25 – Delivered vs Not-Delivered order count

In [ ]:
pd.read_sql_query("""
SELECT
  SUM(CASE WHEN status='Delivered' THEN 1 ELSE 0 END) AS Delivered,
  SUM(CASE WHEN status<>'Delivered' THEN 1 ELSE 0 END) AS Not_Delivered
FROM orders;
""", conn)

,Delivered,Not_Delivered
0,6,4


## Q26 – ACID Properties

### Atomicity
A transaction is treated as a single indivisible unit — it **either completes fully or not at all**. If any step fails, all previous steps in that transaction are rolled back.

### Consistency
A transaction brings the database from one **valid state to another valid state**, always respecting all defined rules, constraints, and cascades.

### Isolation
Concurrent transactions execute **independently** of each other. Intermediate states of a transaction are invisible to other transactions until it commits.

### Durability
Once a transaction is **committed**, the changes are permanently saved even if the system crashes immediately afterward (ensured via write-ahead logs / journaling).

---
### Bank Transfer Example – ₹1000 from Account A to Account B

| Property | How it applies |
|-------------|--------------------------------------------------|
| **Atomicity** | Both the debit from A and the credit to B must succeed. If the credit fails, the debit is rolled back — money is never lost. |
| **Consistency** | The total balance across all accounts remains the same before and after the transfer. |
| **Isolation** | Another transaction checking balances simultaneously sees either the old state or the new state — never the in-between state where money has been debited but not yet credited. |
| **Durability** | Once the transfer is committed and the customer receives confirmation, a system crash cannot undo it. |

## Q27 – Transaction: Place a New Order

The following transaction places order #1011 for customer 102, adds two items, and decrements stock — all atomically.

```sql
BEGIN TRANSACTION;

-- Step 1: Create the order header
INSERT INTO orders
VALUES (1011, 102, CURRENT_DATE, 'Pending', 1898.00);

-- Step 2: Add Bedsheet Set (product 206)
INSERT INTO order_items
VALUES (5016, 1011, 206, 1, 1299.00, 0);

-- Step 3: Add Cushion Covers (product 208)
INSERT INTO order_items
VALUES (5017, 1011, 208, 1, 599.00, 0);

-- Step 4: Reduce stock for product 206
UPDATE products
SET stock_qty = stock_qty - 1
WHERE product_id = 206;

-- Step 5: Reduce stock for product 208
UPDATE products
SET stock_qty = stock_qty - 1
WHERE product_id = 208;

COMMIT;
```

**If any step fails (e.g., stock_qty would go negative):**

```sql
ROLLBACK;
```

The `ROLLBACK` undoes all steps, leaving the database in its original state. This is **Atomicity** in action.

In [ ]:
try:
    cur.execute("BEGIN")
    cur.execute("INSERT INTO orders VALUES (1011, 102, '2024-09-01', 'Pending', 1898.00)")
    cur.execute("INSERT INTO order_items VALUES (5016, 1011, 206, 1, 1299.00, 0)")
    cur.execute("INSERT INTO order_items VALUES (5017, 1011, 208, 1, 599.00, 0)")
    cur.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206")
    cur.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208")
    conn.commit()
    print("Transaction committed successfully.")
except Exception as e:
    conn.rollback()
    print(f"Transaction rolled back due to error: {e}")

# Verify the new order
print(pd.read_sql_query("SELECT * FROM orders WHERE order_id=1011;", conn).to_string(index=False))

Transaction committed successfully.
 order_id  customer_id order_date  status  total_amount
     1011          102 2024-09-01 Pending        1898.0
